### ***LOADING THE WINDOWED AUDIO***

In [ ]:
from pathlib import Path

SAMPLE_RATE: int = 16_000
N_MELS: int = 26
N_MFCC: int = 13
F_MIN: float = 0.0
F_MAX: float = SAMPLE_RATE / 2
ROLLOFF_PERCENT: float = 0.85

FRAME_SIZE: int = 400
N_FFT: int = FRAME_SIZE

BATCH_SIZE: int = 4_096
# arbitrary btw
# batch size is just to save on memory and space
# not necessary but Little Ms. LLM gave it so ok

NOTEBOOK_DIR: Path = Path.cwd()
DIR_REPOSITORY_ROOT: Path = NOTEBOOK_DIR
DIR_INPUT_FFT: Path = DIR_REPOSITORY_ROOT / "preprocessing-03-fft.npy"
DIR_OUTPUT_MFCC: Path = DIR_REPOSITORY_ROOT / "extracted-mfcc.npy"

In [ ]:
import numpy as np

INPUT_FFT:np.ndarray = np.load(DIR_INPUT_FFT.resolve(), allow_pickle=True)

### ***CHECKING THE WINDOWED FRAMES***

note : Only load trusted NumPy files with `allow_pickle=True`. A pickled object array can contain unsafe code.

In [ ]:
# checking the file contents
if INPUT_FFT.ndim != 2 or INPUT_FFT.shape[1] != 5:
    raise ValueError(f"Expected an FFT array with shape (number_of_frames, 5), got {INPUT_FFT.shape}.")
if len(INPUT_FFT) == 0:
    raise ValueError("The FFT input contains no frames.")
EXPECTED_BINS: int = N_FFT // 2 + 1
first_power = np.asarray(INPUT_FFT[0, 4])
if first_power.shape != (EXPECTED_BINS,):
    raise ValueError(
        f"Expected {EXPECTED_BINS} power bins for N_FFT={N_FFT}, got {first_power.shape}."
    )

n = 25
print("="*50, "SHAPE", f"\t{INPUT_FFT.shape=}", sep="\n")
print("SHAPE", f"\t{INPUT_FFT[n].shape=}", sep="\n")
print("="*50, "DIR", f"\t{INPUT_FFT[n][0]=}", sep="\n")
print("="*50, "FRAME#", f"\t{INPUT_FFT[n][1]=}", sep="\n")

print("="*50, "SHAPE", f"\t{INPUT_FFT[n][2].shape=}", sep="\n")
print("FRAME_DATA", f"\t{INPUT_FFT[n][2]=}", sep="\n")

### ***DEFINE OUTPUT COLUMNS***

In [ ]:
feature_columns: list[str] = [
    "zcr_mean",
    "centroid_mean",
    "rolloff_mean",
    "rms_mean",
]
for coefficient in range(1, N_MFCC + 1):
    feature_columns.extend(
        [f"mfcc{coefficient:02}_mean", f"mfcc{coefficient:02}_std"]
    )

output_columns: list[str] = ["dir", "frame #", *feature_columns]
print(f"Output column count: {len(output_columns)}")
print(output_columns)